# GEO Metadata validated and modified process (Gemini-2.0-flash FALSE & Infered has control FALSE)

- ヒトの熱ストレスに関連するRNA-Seqにおいて、Gemini-2.0-flash の判断と自分の判断の結果、解析に使用するに値するデータのメタデータを整理するnotebook
- 整理する際、なぜこのようなデータフレームの処理をするのかについても記載する
- `geofetch`から取得したデータと`iseq`から取得したデータ
- [../GenAI_validate/geo_metadata_genai_validation_false_no_control.tsv](../../Data/curation/GenAI_validate/20250504/geo_metadata_genai_validation_false_no_control.tsv) に分類されたものについてここに記載する

- **いくつかのデータは使用できそうなので収集しておく**

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(100)
pl.Config.set_tbl_rows(4)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera",
    "tax_lineage",
    "bam_md5",
    "fastq_md5",
    "sra_md5"
]

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. GSE124510 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-05-04
- [Proteasome inhibitor-induced modulation reveals the spliceosome as a specific therapeutic vulnerability in multiple myeloma](https://doi.org/10.1038/s41467-020-15521-4)
- `bortezomib`などのデータをフィルタリングし、コントロールと `42C for 4 h`のデータを収集

In [4]:
ID = "GSE124510"

GSE124510_PEP = pl.read_csv(
    f"../../Data/metadata/metadata_geofetch/{ID}/{ID}_PEP/{ID}_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "RNA-Seq")
).filter(
    (pl.col("sample_title").str.contains(r'MM.1S 0 h')) |
    (pl.col("sample_title").str.contains(r'MM.1S heat shock'))
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Hyper Prep RNA sequencing kit").alias("rna_library"), # add rna_library column from GEO webpage (KAPA Biosystems)
    pl.lit("https://doi.org/10.1038/s41467-020-15521-4").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("240 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature"), # add control temperature column
    pl.lit("MM.1S").alias("cell_line")

).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE124510 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE124510_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE124510_iseq_filtered = GSE124510[[s.name for s in GSE124510 if not (s.null_count() == GSE124510.height)]]

GSE124510_combined = GSE124510_PEP.join(
    GSE124510_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE124510_combined = GSE124510_combined[[s.name for s in GSE124510_combined if not (s.null_count() == GSE124510_combined.height)]]

if GSE124510_combined is not None:
    GSE124510_combined.write_csv(
        "../../Data/metadata/metadata_modified/GSE124510_modified_metadata.tsv",
        separator='\t'
    )

# display(GSE124510_combined)

In [5]:
defined_pairs_gse124510 = [
    ("SRR10761036", "SRR10761033"), # MM.1S heat shock 4 h mRNA replicate 1 vs MM.1S 0 h control mRNA replicate 1
    ("SRR10761037", "SRR10761034"), # MM.1S heat shock 4 h mRNA replicate 2 vs MM.1S 0 h control mRNA replicate 2
    ("SRR10761038", "SRR10761035")  # MM.1S heat shock 4 h mRNA replicate 3 vs MM.1S 0 h control mRNA replicate 3
]

paired_metadata_df_gse124510 = create_paired_metadata(
    combined_df=GSE124510_combined,
    defined_sample_pairs=defined_pairs_gse124510,
    sample_id_column_name="accession" # default value, but specified to clarify
)


if paired_metadata_df_gse124510 is not None:
    paired_metadata_df_gse124510.write_csv(
        "../../Data/metadata/metadata_paired/GSE124510_paired_metadata.tsv",
        separator = '\t'
    )
    
    display(paired_metadata_df_gse124510)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,experiment,timepoint,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR10761036""","""SRR10761033""","""mm_1s_heat_shock_4_h_mrna_replicate_1 vs mm_1s_0_h_control_mrna_replicate_1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR10761036 vs SRR10761033""","""SRX7435253 vs SRX7435250""","""MM.1S heat shock 4 h mRNA replicate 1 vs MM.1S 0 h control mRNA replicate 1""","""GSM4231707 vs GSM4231704""","""Public on Feb 18 2020""","""Dec 23 2019""","""Feb 18 2020""","""SRA""","""1""","""MM.1S myeloma cell line""","""Homo sapiens""","""9606""","""For heat shock samples, MM.1S cultures were incubated at 42C for 4 h in 3 biological replicates. Timecourse MM.1S cultures were treated with 30 nM carfilzomib and harvested after 0, 8, 16, or 24 h tr…","""Heat shock experiment and Timecourse MM.1S cultures were grown in RPMI-1640, 10% FBS, 1% Pencillin-Streptomycin. Single time point MM.1S and AMO-1 cultures were used for concurrent global proteomics …","""polyA RNA""","""juncbase_tables_DMSO_vs_heatshock_jcn272.tsv vs juncbase_tables_DMSO_vs_heatshock_jcn272.tsv, juncbase_tables_DMSO_vs_Bortezomib_jcn272.tsv""","""GPL20301""","""Hector,Han-Li,Huang""","""Wiita Lab""","""Laboratory Medicine""","""University of California San Francisco""","""185 Berry St, Ste 290/Rm 2410""","""San Francisco""","""CALIFORNIA""","""94107""","""USA""","""Illumina HiSeq 4000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM4231707 vs GSM4231704""","""NONE""","""GSE124510""","""0""","""SRX7435253 vs SRX7435250""","""MM.1S""","""heat shock induced intron retention""","""4 h vs 0 h (untreated)""","""UCSC hg19""","""""delink_c2g_fw5rv_xpr.txt"" in-house C++ scripts output with UCSC Gene ID, raw reads, and normalized RPKM. Conversion tables available from UCSC through the kgXref function were used to convert UCSC …","""SAMN13673568 vs SAMN13673597""","""Hyper Prep RNA sequencing kit""","""https://doi.org/10.1038/s41467-020-15521-4""","""42 C""","""240 min.""","""37 C (inferred)""","""SRR10761036 vs SRR10761033""","""Illumina HiSeq 4000 sequencing: GSM4231707: MM.1S heat shock 4 h mRNA replicate 1 Homo sapiens RNA-Seq vs Illumina HiSeq 4000 sequencing: GSM4231704: MM.1S 0 h control mRNA

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. GSE194098 (SINGLE,PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-04
- [HSF2 cooperates with HSF1 to drive a transcriptional program critical for the malignant state](https://doi.org/10.1126/sciadv.abj6526)
- HSF1,2のKDなどのデータが含まれている
- 熱ショックが加えられているデータを発見！
- フィルタリングしてデータを収集
- `we performed RNA-seq in our HSF2 knockout MDA-MB-231 cell lines grown either at 37°C or following a 42°C heat shock along with wild-type and HSF1 knockout control lines.`という記述あり
- Supplementary Figure 7. に以下のような記載がある
- `Gene expression changes of heat shock (HS) treated (42°C, 1 hour) MDA-MB-231 cells with either HSF1 (sgHSF1) or HSF2 (sgHSF2) knockout, expressed as log2 fold-change (L2FC) relative to non-targeting sgRNA controls without heat shock.`

In [6]:
GSE194098_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE194098/GSE194098_PEP/GSE194098_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "RNA-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).filter(
    (pl.col("sample_library_strategy") == "RNA-Seq") &
    ((pl.col("sample_title").str.contains(r'MDAMB231_sgHSF1_Control_')) |
     (pl.col("sample_title").str.contains(r'MDAMB231_sgHSF2_Control_')) |
     (pl.col("sample_title").str.contains(r'MDAMB231_sgHSF1_HeatShock_')) |
     (pl.col("sample_title").str.contains(r'MDAMB231_sgHSF2_HeatShock_')) |
     (pl.col("sample_title").str.contains(r'MDAMB231_WT_HeatShock_')) |
     (pl.col("sample_title").str.contains(r'MDAMB231_WT_Control_'))
     )
).with_columns(
    pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession"),
    pl.lit("Lexogen QuantSeq FWD Kit").alias("rna_library"), # add rna_library column from GEO webpage (Lexogen)
    pl.lit("https://doi.org/10.1126/sciadv.abj6526").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE194098 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE194098_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE194098_iseq_filtered = GSE194098[[s.name for s in GSE194098 if not (s.null_count() == GSE194098.height)]]

GSE194098_combined = GSE194098_PEP.join(
    GSE194098_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE194098_combined = GSE194098_combined[[s.name for s in GSE194098_combined if not (s.null_count() == GSE194098_combined.height)]]

if GSE194098_combined is not None:
    GSE194098_combined.write_csv(
        "../../Data/metadata/metadata_modified/GSE194098_modified_metadata.tsv",
        separator='\t'
    )

# display(GSE194098_combined)

In [7]:
defined_pairs_gse194098 = [
   ("SRR17682123", "SRR17682125"), # MDAMB231_sgHSF1_HeatShock_A vs MDAMB231_sgHSF1_Control_A
   ("SRR17682122", "SRR17682124"), # MDAMB231_sgHSF1_HeatShock_B vs MDAMB231_sgHSF1_Control_B
   ("SRR17682203", "SRR17682121"), # MDAMB231_sgHSF2_HeatShock_A vs MDAMB231_sgHSF2_Control_A
   ("SRR17682201", "SRR17682120"), # MDAMB231_sgHSF2_HeatShock_B vs MDAMB231_sgHSF2_Control_B
   ("SRR17682198", "SRR17682200"), # MDAMB231_WT_HeatShock_A vs MDAMB231_WT_Control_A
   ("SRR17682197", "SRR17682199")  # MDAMB231_WT_HeatShock_B vs MDAMB231_WT_Control_B
]

paired_metadata_df_gse194098 = create_paired_metadata(
    combined_df=GSE194098_combined,
    defined_sample_pairs=defined_pairs_gse194098,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse194098 is not None:
    paired_metadata_df_gse194098.write_csv(
        "../../Data/metadata/metadata_paired/GSE194098_paired_metadata.tsv",
        separator = '\t'
    )

    display(paired_metadata_df_gse194098)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,vector,sirna_sgrna,antibody,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR17682123""","""SRR17682125""","""mdamb231_sghsf1_heatshock_a vs mdamb231_sghsf1_control_a""","""MDAMB231_sgHSF1_HeatShock_A vs MDAMB231_sgHSF1_Control_A""","""GSM5828056 vs GSM5828054""","""Public on Mar 15 2022""","""Jan 20 2022""","""Mar 15 2022""","""SRA""","""1""","""cancer cell line""","""Homo sapiens""","""9606""","""HeatShock, See sample section for treatment vs Control, See sample section for treatment""","""Cells were grown at 37°C and 5% CO2""","""total RNA""","""Lexogen 3' QuantSeq FWD for Illumina, RNA was extracted using the Qiagen RNeasy kit using the Qiashredder for homogenization of cells grown in vitro and a rotor-stator homogenizer for xenograft tumors""","""MDAMB231_HS_htseq.all.counts.txt""","""GPL18573""","""Marc,,Mendillo""","""mendillo@northwestern.edu""","""Biochemistry and Molecular Genetics""","""303 E Superior St""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NextSeq 500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5828056 vs GSM5828054""","""NONE""","""GSE194098""","""0""","""SRX13845434 vs SRX13845432""","""MDAMB231""","""N/A""","""sgHSF1.1""","""N/A""","""hg38""","""SAMN25144533 vs SAMN25144535""","""Lexogen QuantSeq FWD Kit""","""https://doi.org/10.1126/sciadv.abj6526""","""42 C""","""60 min.""","""37 C""","""SRR17682123 vs SRR17682125""","""NextSeq 500 sequencing: GSM5828056: MDAMB231_sgHSF1_HeatShock_A Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM5828054: MDAMB231_sgHSF1_Control_A Homo sapiens RNA-Seq""","""HSF2 cooperates with HSF1 to drive a transcriptional program critical for the malignant state""","""SRA1359445""","""cDNA""","""2022-03-18""","""135030960 vs 118141619""","""167017520 vs 146770862""","""ILLUMINA""","""Generic""","""MDAMB231_sgHSF1_HeatShock_A vs MDAMB231_sgHSF1_Control_A""","""SRP356009""","""3893070 vs 3417476""","""HSF2 cooperates with HSF1 to drive a transcriptional program critical for the malignant state""","""MDAMB231_sgHSF1_HeatShock_A vs MDAMB231_sgHSF1_Control_A""","""NextSeq 500""","""NextSeq 500 sequencing: GSM5828056: MDAMB231_sgHSF1_HeatShock_A Homo sapiens RNA-Seq vs NextSeq 500 sequencing: GSM5828054: MDAMB231_sgHSF1_Control_A Homo sapiens RNA-Seq""","""288681297 vs 253739045""","""RNA-Seq""","

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE127890 (SINGLE) Nanopore

- validate date 2025-05-04
- [Native molecule sequencing by nano-ID reveals synthesis and stability of RNA isoforms](https://doi.org/10.1101/gr.257857.119)
- 以前の研究でも使用しているnanoporeのデータで、RNAのisoformを検出するための手法 nano-ID (nanopore sequencing–based isoform dynamics) を開発した論文
- `To show the advantages of nano-ID, we subjected human K562 cells to heat shock (42°C) for 60 min in the presence of 5EU (5EU 60 min HS) (Fig. 4A).`
- 上記のような記述に関するデータを収集
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

#### GSE234617 (SINGLE) NO DOWLOAD

- validate date 2025-05-06
- [AFF4 globally affects the release of paused RNA polymerase II in HEL cells](https://pubmed.ncbi.nlm.nih.gov/37609817/)
- この論文の全文にアクセスすることができなかった
- また、メタデータを見てもこのデータには熱ストレスではなく、AFF4をノックダウンしているデータが見られたため、今回はこの論文は収集しない

In [8]:
# GSE234617_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE234617/GSE234617_PEP/GSE234617_PEP_raw.csv",
#     separator=','
# )

# display(GSE234617_PEP)

#### GSE240883 (SINGLE) NO DOWNLOAD

- validate date 2025-05-04
- [Phosphorylation of a nuclear condensate regulates cohesion and mRNA retention](https://doi.org/10.1038/s41467-024-55469-3)
- 熱ストレスには特に関連していないデータのみだったので、今回は解析に使用しない

In [9]:
# GSE240883_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE240883/GSE240883_PEP/GSE240883_PEP_raw.csv",
#     separator=','
# )

# display(GSE240883_PEP)

#### GSE262079 (SINGLE) NO DOWNLOAD

- validate date 2025-05-06
- [First-in-class mitogen-activated protein kinase (MAPK) p38α: MAPK-activated protein kinase 2 dual signal modulator with anti-inflammatory and endothelial-stabilizing properties](https://doi.org/10.1124/jpet.124.002281)
- hyperthermia (FRH (febrile-range hyperthermia)) に関する研究
- `p38 inhibitor, SB203580`処理のデータと、`SF7044(GEn1124)/TNFa` 処理のデータ、`UM101`処理のデータ、DMSO処理のデータが含まれている
- hyperthermiaに関するデータは含まれていないので今回は収集しない

In [10]:
# GSE262079_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE262079/GSE262079_PEP/GSE262079_PEP_raw.csv",
#     separator=','
# )

# display(GSE262079_PEP)

#### GSE240891 (SINGLE) NO DOWNLOAD

- validate date 2025-05-06
- [Phosphorylation of a nuclear condensate regulates cohesion and mRNA retention](https://doi.org/10.1038/s41467-024-55469-3)
- 熱ショックに関する実験は行われているが、RNA-seqに関するデータ,このGEOのデータは`biotinylated RNA`に関するデータ
- 今回は収集しない

In [11]:
# GSE240891_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE240891/GSE240891_PEP/GSE240891_PEP_raw.csv",
#     separator=','
# )

# display(GSE240891_PEP)

#### GSE137591 (SINGLE) NO DOWNLOAD

- validate date 2025-05−06
- [HSF1 Activation Can Restrict HIV Replication](https://doi.org/10.1021/acsinfecdis.0c00166)
- `single-colony human T lymphocyte (CEM) cell line` と `constitutively active variant of HSF1 (cHSF1)` と`18h 1 ug/ml Doxycycline (dox) treatment`に関連するデータ
- 論文中では熱ショック処理実験をしているが、GEOに登録されているデータには熱ショック関連のデータがない
- 今回は収集しない

In [12]:
# GSE137591_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE137591/GSE137591_PEP/GSE137591_PEP_raw.csv",
#     separator=','
# )

# display(GSE137591_PEP)

#### GSE202189 (SINGLE) NO DOWNLOAD

- validate date 2025-05-06
- [SPT6 functions in transcriptional pause/release via PAF1C recruitment](https://doi.org/10.1016/j.molcel.2022.06.037)
- SPT6 と [AID (activation-induced cytidine deaminase)](https://www.yodosha.co.jp/jikkenigaku/keyword/index.html?id=1821) を導入しているデータ
- 論文中では熱ショックをかけている記述があるが、GEOに登録されているデータではこのデータがないため、今回は収集しない

In [13]:
# GSE202189_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE202189/GSE202189_PEP/GSE202189_PEP_raw.csv",
#     separator=','
# )

# display(GSE202189_PEP)

#### GSE208005 （SINGLE） NO DOWNLOAD

- validation date: 2025-05-06
- [Co-targeting HSP90 alpha and CDK7 overcomes resistance against HSP90 inhibitors in BCR-ABL1+ leukemia cells](https://doi.org/10.1038/s41419-023-06337-3)
- HSP90のノックアウトや、阻害剤のPU-H71(Zelavespib)などが処理されているデータ
- 熱ストレスに関するデータは含まれていないため、今回は収集しない

In [14]:
# GSE208005_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE208005/GSE208005_PEP/GSE208005_PEP_raw.csv",
#     separator=','
# )

# display(GSE208005_PEP)

### GSE202187 (SINGLE) PRO-seq

- validation date: 2025-05-06
- [SPT6 functions in transcriptional pause/release via PAF1C recruitment](https://doi.org/10.1016/j.molcel.2022.06.037)
- GSE202189に関連するデータで、PRO-seqのデータ
- PRO-seqとSPT6に対するauxin-inducible degradation (AID) systemが行われている論文
- 42℃, 60min.
- nacsent RNAのデータが使えるかわからないが、念の為収集
- 処理を[12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)に移動

#### GSE181161 (SINGLE) NO DOWNLOAD

- validation date: 2025-05−06
- [Quantifying RNA synthesis at rate-limiting steps of transcription using nascent RNA-sequencing data](https://doi.org/10.1016/j.xpro.2021.101036)
- PRO-seqのデータで、conditionカラムをみると `Unstressed (no heat shock)`のデータのみ含まれているため、今回は収集しない

In [15]:
# GSE181161_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE181161/GSE181161_PEP/GSE181161_PEP_raw.csv",
#     separator=','
# )

# display(GSE181161_PEP)

#### GSE165211 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-06
- [Cancer-Associated Fibroblasts Promote Aggressive Gastric Cancer Phenotypes via Heat Shock Factor 1–Mediated Secretion of Extracellular Vesicles](https://doi.org/10.1158/0008-5472.CAN-20-2756)
- HSF1に関連した論文だが、熱ショックに関連したデータは含まれていなかったため、今回は収集しない

In [16]:
# GSE165211_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE165211/GSE165211_PEP/GSE165211_PEP_raw.csv",
#     separator=','
# )

# display(GSE165211_PEP)

#### GSE164717 (SINGLE) NO DOWNLOAD

- validation date 2025-05−06
- [Inhibition of HSP90 as a Strategy to Radiosensitize Glioblastoma: Targeting the DNA Damage Response and Beyond](https://doi.org/10.3389/fonc.2021.612354)
- `The human GBM cell lines LN229 and T98G` に関して、HSP90 inhibition 処理のデータをかけているデータ
- 今回はこのデータを収集しない

In [17]:
# GSE164717_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE164717/GSE164717_PEP/GSE164717_PEP_raw.csv",
#     separator=','
# )

# display(GSE164717_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. GSE180253 (SINGLE) RNA-seq

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-06
- [m6Am-seq reveals the dynamic m6Am methylation in the human transcriptome](https://doi.org/10.1038/s41467-021-25105-5)
- m6Am-seq を行うための、`Input` として使用されている RNA-seq データを収集する
- `heat_shock_` と `Untreated_` のデータを念の為収集する
- bulk RNA-seq以外のデータの処理を[./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に移動

In [18]:
GSE180253_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE180253/GSE180253_PEP/GSE180253_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("sample_library_strategy") == "RNA-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).filter(
    ((pl.col("sample_title").str.contains(r'heat_shock_')) | 
     (pl.col("sample_title").str.contains(r'Untreated_'))) & # remove HSF1 knockdown data
    ~(pl.col("sample_title").str.contains(r'Hypoxia_')) # remove HSF1 knockdown data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("SMARTer® Stranded Total RNA-Seq Kit v2—Pico Input Mammalian (634413, Takara – Clontech, Japan)").alias("rna_library"), # add rna_library column from GEO page
    pl.lit("https://doi.org/10.1038/s41467-021-25105-5").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE180253 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE180253_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE180253_iseq_filtered = GSE180253[[s.name for s in GSE180253 if not (s.null_count() == GSE180253.height)]]


GSE180253_combined_rna_seq = GSE180253_PEP.join(
    GSE180253_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE180253_combined_rna_seq = GSE180253_combined_rna_seq[[s.name for s in GSE180253_combined_rna_seq if not (s.null_count() == GSE180253_combined_rna_seq.height)]]

GSE180253_combined_rna_seq.write_csv(
    "../../Data/metadata/metadata_modified/GSE180253_rna_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE180253_combined_rna_seq)

In [19]:
# RNA-seq
defined_pairs_gse180253_rna_seq = [
    ("SRR15177495", "SRR15177509") # heat_shock_input vs Untreated_input
]

paired_metadata_df_gse180253_rna_seq = create_paired_metadata(
    combined_df=GSE180253_combined_rna_seq, 
    defined_sample_pairs=defined_pairs_gse180253_rna_seq,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse180253_rna_seq is not None:
    paired_metadata_df_gse180253_rna_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE180253_rna_seq_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse180253_rna_seq)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR15177495""","""SRR15177509""","""heat_shock_input vs untreated_input""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR15177495 vs SRR15177509""","""SRX11484753 vs SRX11484767""","""heat_shock_input vs Untreated_input""","""GSM5456689 vs GSM5456703""","""Public on Jul 19 2021""","""Jul 16 2021""","""Jul 20 2021""","""SRA""","""1""","""HEK293T""","""Homo sapiens""","""9606""","""polyA RNA""","""RNA-seq""","""GPL20795""","""Kai,,Li""","""li_kai@pku.edu.cn""","""Yi Lab""","""School of Life Science""","""Peking University""","""Yi He Yuan Street No.5""","""BeiJing""","""100871""","""China""","""HiSeq X Ten""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5456689 vs GSM5456703""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5456nnn/GSM5456689/suppl/GSM5456689_HS-input_S9_L008_genome.bam_downsample.sort.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM5456nnn/GSM5456703/suppl/GSM54…","""GSE180253""","""0""","""SRX11484753 vs SRX11484767""","""HEK293T""","""human embryonic kidney""","""heat_shock_input vs Untreated_input""","""hg19, UCSC Genome Browser""","""bigWig files, csv files""","""SAMN20284029 vs SAMN20284049""","""SMARTer® Stranded Total RNA-Seq Kit v2—Pico Input Mammalian (634413, Takara – Clontech, Japan)""","""https://doi.org/10.1038/s41467-021-25105-5""","""43 C""","""120 min.""","""37 C""","""SRR15177495 vs SRR15177509""","""HiSeq X Ten sequencing: GSM5456689: heat_shock_input Homo sapiens RNA-Seq vs HiSeq X Ten sequencing: GSM5456703: Untreated_input Homo sapiens RNA-Seq""","""m6Am-seq reveals the dynamic m6Am methylation in the human transcriptome""","""SRA1260268""","""cDNA""","""2021-07-24""","""636204753 vs 721673644""","""821876256 vs 916083356""","""ILLUMINA""","""Generic""","""heat_shock_input vs Untreated_input""","""SRP328696""","""11372826 vs 13371894""","""m6Am-seq reveals the dynamic m6Am methylation in the human transcriptome""","""heat_shock_input vs Untreated_input""","""HiSeq X Ten""","""HiSeq X Ten sequencing: GSM5456689: heat_shock_input Homo sapiens RNA-Seq vs HiSeq X Ten sequencing: GSM5456703: Untreated_input Homo sapiens RNA-Seq""","""human embryonic kidney""","""1705923900 vs 200578410

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. GSE145525 (SINGLE)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-06
- [Acute perturbation strategies in interrogating RNA polymerase II elongation factor function in gene expression](https://doi.org/10.1101/gad.346106.120)
- mRNA-seqとChIP-seqのデータが含まれている、RNA polymerase IIに関連したデータ
- BRD4に対してAIDシステムを行っている + 42℃ 60min.を行っているデータなどが含まれている
- mRNA-seqのデータを収集する

In [20]:
GSE145525_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE145525/GSE145525_PEP/GSE145525_PEP_raw.csv",
    separator=','
).filter(
    (pl.col("protocol") == "cDNA") &
    (pl.col("sample_library_strategy") == "RNA-Seq") &
    (pl.col("sample_organism_ch1") == "Homo sapiens")
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("NEBNext Ultra II Directional RNA library preparation kit").alias("rna_library"), # add rna_library column from GEO page / Article
    pl.lit("https://doi.org/10.1101/gad.346106.120").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("240 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE145525 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE145525_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE145525_iseq_filtered = GSE145525[[s.name for s in GSE145525 if not (s.null_count() == GSE145525.height)]]

GSE145525_combined = GSE145525_PEP.join(
    GSE145525_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE145525_combined = GSE145525_combined[[s.name for s in GSE145525_combined if not (s.null_count() == GSE145525_combined.height)]]

GSE145525_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE145525_modified_metadata.tsv",
    separator='\t'
)

# display(GSE145525_combined)

In [21]:
defined_pairs_gse145525 = [
    ("SRR12926161", "SRR12926157"), # 2hH2O_4h42C_BRD4AID_rep1 vs 2hH2O_4h37C_BRD4AID_rep1
    ("SRR12926162", "SRR12926158"), # 2hH2O_4h42C_BRD4AID_rep1 vs 2hH2O_4h37C_BRD4AID_rep1
    ("SRR12926163", "SRR12926159"), # 2hH2O_4h42C_BRD4AID_rep2 vs 2hH2O_4h37C_BRD4AID_rep2
    ("SRR12926164", "SRR12926160"), # 2hH2O_4h42C_BRD4AID_rep2 vs 2hH2O_4h37C_BRD4AID_rep2
    ("SRR12926169", "SRR12926165"), # 2hAux_4h42C_BRD4AID_rep1 vs 2hAux_4h37C_BRD4AID_rep1
    ("SRR12926170", "SRR12926166"), # 2hAux_4h42C_BRD4AID_rep1 vs 2hAux_4h37C_BRD4AID_rep1
    ("SRR12926171", "SRR12926167"), # 2hAux_4h42C_BRD4AID_rep2 vs 2hAux_4h37C_BRD4AID_rep2
    ("SRR12926172", "SRR12926168"), # 2hAux_4h42C_BRD4AID_rep2 vs 2hAux_4h37C_BRD4AID_rep2
    ("SRR12926177", "SRR12926173"), # 2hH2O_4h42C_DoubleAID_rep1 vs 2hH2O_4h37C_DoubleAID_rep1
    ("SRR12926178", "SRR12926174"), # 2hH2O_4h42C_DoubleAID_rep1 vs 2hH2O_4h37C_DoubleAID_rep1
    ("SRR12926179", "SRR12926175"), # 2hH2O_4h42C_DoubleAID_rep2 vs 2hH2O_4h37C_DoubleAID_rep2
    ("SRR12926180", "SRR12926176"), # 2hH2O_4h42C_DoubleAID_rep2 vs 2hH2O_4h37C_DoubleAID_rep2
    ("SRR12926185", "SRR12926181"), # 2hAux_4h42C_DoubleAID_rep1 vs 2hAux_4h37C_DoubleAID_rep1
    ("SRR12926186", "SRR12926182"), # 2hAux_4h42C_DoubleAID_rep1 vs 2hAux_4h37C_DoubleAID_rep1
    ("SRR12926187", "SRR12926183"), # 2hAux_4h42C_DoubleAID_rep2 vs 2hAux_4h37C_DoubleAID_rep2
    ("SRR12926188", "SRR12926184")  # 2hAux_4h42C_DoubleAID_rep2 vs 2hAux_4h37C_DoubleAID_rep2
]

paired_metadata_df_gse145525 = create_paired_metadata(
    combined_df=GSE145525_combined,
    defined_sample_pairs=defined_pairs_gse145525,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse145525 is not None:
    paired_metadata_df_gse145525.write_csv(
        "../../Data/metadata/metadata_paired/GSE145525_paired_metadata.tsv",
        separator="\t"
    )

    display(paired_metadata_df_gse145525)
else:
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_growth_protocol_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,genotype,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR12926161""","""SRR12926157""","""2hh2o_4h42c_brd4aid_rep1 vs 2hh2o_4h37c_brd4aid_rep1""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""multi""","""SRX9390309 vs SRX9390307""","""2hH2O_4h42C_BRD4AID_rep1 vs 2hH2O_4h37C_BRD4AID_rep1""","""GSM4871709 vs GSM4871707""","""Public on Jan 15 2021""","""Oct 28 2020""","""Jan 15 2021""","""SRA""","""1""","""DLD-1""","""Homo sapiens""","""9606""","""Cells were treated with 500 µM Auxin (3-indole-acetic acid sodium salt, abcam), 250 nM dBET6 (AOBIOUS), or 20 nM KL-1 (Liang et al. 2018). Heat shock treatment at 42 ˚C for 1 h was carried out as desc…","""DLD-1 or HCT116 cells were cultured in DMEM (Corning) supplemented with 10% FBS (Corning), 1X GlutaMAX (Gibco), 100 U/ml Penicillin and 100 µg/ml Streptomycin (Gibco).""","""total RNA""","""For ChIP-seq, 20-50 million cells were crosslinked with 1% paraformaldehyde (Electron Microscopy Sciences) in PBS for 10 min at r.t. Chromatin was sonicated with the Covaris E220 for 4 min with 10% du…","""For ChIP-seq and RNA-seq, single-end raw reads were trimmed by Trimmomatic 0.33 (Bolger et al., 2014) and aligned to hg19 using Bowtie (1.1.2) and TopHat (2.1.0)(Langmead et al. 2009; Trapnell et al. …","""GPL24676""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM4871709 vs GSM4871707""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4871nnn/GSM4871709/suppl/GSM4871709_2hH2O_4h42C_BRD4AID_rep1_minus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4871nnn/GSM4871707/suppl/GSM4871707_2hH2O_4…","""GSE145525""","""0""","""SRX9390309 vs SRX9390307""","""Colorectal adenocarcinoma""","""BRD4-AID OsTIR1""","""hg19""","""SAMN16582396 vs SAMN16582398""","""NEBNext Ultra II Directional RNA library preparation kit""","""https://doi.org/10.1101/gad.346106.120""","""42 C""","""240 min.""","""37 C (inferred)""","""SRR12926161 vs SRR12926157""","""Illumina NovaSeq 6000 sequencing: GSM4871709: 2hH2O_4h42C_BRD4AID_rep1 Ho

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

#### GSE148659 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-06
- [Allosteric HSP70 inhibitors perturb mitochondrial proteostasis and overcome proteasome inhibitor resistance in multiple myeloma](https://doi.org/10.1016/j.chembiol.2022.06.010)
- HSP70の阻害剤に対する解析を行っている
- データを見た所、熱ストレスをかけているデータは含まれていなかったので今回は収集しない

In [22]:
# GSE148659_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE148659/GSE148659_PEP/GSE148659_PEP_raw.csv",
#     separator=','
# )

# display(GSE148659_PEP)

#### GSE143743 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-06
- [Knocking out C9ORF72 Exacerbates Axonal Trafficking Defects Associated with Hexanucleotide Repeat Expansion and Reduces Levels of Heat Shock Proteins](https://doi.org/10.1016/j.stemcr.2020.01.010)
- HSPに関連している論文だが、C9ORF72 knockout (KO) に主に焦点を当てている論文
- 熱ストレスに関連するデータは含まれていなかったので、今回は収集しない

In [23]:
# GSE143743_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE143743/GSE143743_PEP/GSE143743_PEP_raw.csv",
#     separator=','
# )

# display(GSE143743_PEP)

#### GSE125943 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-06
- [Dual RNA-Seq of Human Leprosy Lesions Identifies Bacterial Determinants Linked to Host Immune Response](https://doi.org/10.1016/j.celrep.2019.02.109)
- HSPに関する記述はあるものの、ハンセン病の感染に関する研究
- メタデータを見ても熱ストレスに関連するデータはないため、今回は収集しない

In [24]:
# GSE125943_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE125943/GSE125943_PEP/GSE125943_PEP_raw.csv",
#     separator=','
# )

# display(GSE125943_PEP)

#### GSE90715 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-06
- [Oncogenic hijacking of the stress response machinery in T cell acute lymphoblastic leukemia](https://doi.org/10.1038/s41591-018-0105-8)
- CUTLL1 cellに対して、HSF1のノックダウンなどが行われている研究
- 熱ストレスに関連したデータは含まれていないため、今回は収集しない

In [25]:
# GSE90715_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE90715/GSE90715_PEP/GSE90715_PEP_raw.csv",
#     separator=','
# )

# display(GSE90715_PEP)

#### GSE96869 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-06
- [Nascent RNA sequencing reveals a dynamic global transcriptional response at genes and enhancers to the natural medicinal compound celastrol](https://doi.org/10.1101/gr.222935.117)
- Celastrol (also known as tripterine) 処理を行っているデータ
- 全てのデータがcelastrol処理だったので今回は収集しない

### GSE70485 (SINGLE) m1A-seq

- validation date: 2025-05-06
- [The dynamic N1-methyladenosine methylome in eukaryotic messenger RNA](https://doi.org/10.1038/nature16998)
- m1A-seqを行っているデータ
- 熱ショックを加えているため、念の為収集する
- `HEPG2 mRNA heat shock 4h IP`, `HEPG2 mRNA heat shock 4h input`, `hepg2_mrna_untreated_ip_replicate_b`, `hepg2_mrna_untreated_input_replicate_b`の4つを念の為収集
- ※ 他にもreplicate C~E があるが今回は使用しない
- 処理を[./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に移動

#### GSE58464 (SINGLE) NO DOWNLOAD

- validation date: 2025-05-06
- [Rapid Isolation of Extracellular Vesicles from Cell Culture and Biological Fluids Using a Synthetic Peptide with Specific Affinity for Heat Shock Proteins](https://doi.org/10.1371/journal.pone.0110443)
- HSPに言及されているが、メタデータには関連したデータがないため、今回は収集しない

In [26]:
# GSE58464_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE58464/GSE58464_PEP/GSE58464_PEP_raw.csv",
#     separator=','
# )

# display(GSE58464_PEP)

#### GSE254828 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Metabolic and proteomic profiling identifies stress response and impaired neural stem cell functions associated with sleep restriction in women](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi)
- 論文が見つからなかった
- 登録されているデータとしても以下のように書いてある
- `RNA-seq of untreated and glutamic acid or methionine treated FUCCI-hNSCs for 30 days in neural expansion medium.`
- 熱ストレスに関連したデータではないため、今回は収集しない

In [27]:
# GSE254828_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE254828/GSE254828_PEP/GSE254828_PEP_raw.csv",
#     separator=','
# )

# # display(GSE254828_PEP)

#### GSE282972 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [An in vitro pharmacogenomic approach reveals subtype-specific therapeutic vulnerabilities in atypical teratoid/rhabdoid tumors (AT/RT)](https://doi.org/10.1016/j.phrs.2025.107660)
- HSPに言及されているが、薬剤スクリーニングに関して研究している論文
- 今回は収集しない

In [28]:
# GSE282972_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE282972/GSE282972_PEP/GSE282972_PEP_raw.csv",
#     separator=','
# )

# # display(GSE282972_PEP)

#### GSE191244 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [HSP70 binds to specific non-coding RNA and regulates human RNA polymerase III](https://doi.org/10.1016/j.molcel.2024.01.001)
- HSPに言及されているが、熱ストレス処理は加えられていないので今回は収集しない

In [29]:
# GSE191244_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE191244/GSE191244_PEP/GSE191244_PEP_raw.csv",
#     separator=','
# )

# # display(GSE191244_PEP)

#### GSE249916 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Proteostasis perturbation of N-Myc leveraging HSP70 mediated protein turnover improves treatment of neuroendocrine prostate cancer](https://doi.org/10.1038/s41467-024-50459-x)
- Hsp70阻害剤であるJG231処理に関するデータが含まれているため、今回は収集しない

In [30]:
# GSE249916_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE249916/GSE249916_PEP/GSE249916_PEP_raw.csv",
#     separator=','
# )

# # display(GSE249916_PEP)

#### GSE249917 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Proteostasis perturbation of N-Myc leveraging HSP70 mediated protein turnover improves treatment of neuroendocrine prostate cancer](https://doi.org/10.1038/s41467-024-50459-x)
- UCD1172 cell lineに対するデータで、熱ストレス処理のデータは含まれていないため、今回は収集しない

In [31]:
# GSE249917_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE249917/GSE249917_PEP/GSE249917_PEP_raw.csv",
#     separator=','
# )

# # display(GSE249917_PEP)

#### GSE267811 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [GRP94 promotes anoikis resistance and peritoneal metastasis through YAP/TEAD1 pathway in gastric cancer](https://doi.org/10.1016/j.isci.2024.110638)
- genotypeを見ると、Wild type と GRP94 のノックダウンのデータが含まれており、熱ストレス処理に関連するデータは含まれていないため、今回は収集しない

In [32]:
# GSE267811_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE267811/GSE267811_PEP/GSE267811_PEP_raw.csv",
#     separator=','
# )

# # display(GSE267811_PEP)

#### GSE276245 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Polo-like kinase 2 promotes microglial activation via regulation of the HSP90α/IKKβ pathway](https://doi.org/10.1016/j.celrep.2024.114827)
- メタデータを見ると、PLK2の過剰発現データが含まれており、熱ストレス曝露処理のデータは含まれており、今回は収集しない

In [33]:
# GSE276245_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE276245/GSE276245_PEP/GSE276245_PEP_raw.csv",
#     separator=','
# )

# # display(GSE276245_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE254232 (PAIRED) NO DOWNLOAD

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-05-06
- validate date: 2026-05-04
- [Unraveling the independent role of METTL3 in m6A modification and tumor progression in esophageal squamous cell carcinoma](https://doi.org/10.1038/s41598-024-64517-3)
- おそらく上記の論文 (このアクセッションが記載されている)
- METTL3,METTL14のノックダウン + 熱ストレスなどが行われている研究
- METTL14のノックダウン + 熱ストレスのデータが含まれていなかったのでこのデータをフィルタリングする
- **`heat shock` とメタデータには記載があるものの、温度、曝露時間が不明なため、収集しない方針とする**

In [34]:
# GSE254232_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE254232/GSE254232_PEP/GSE254232_PEP_raw.csv",
#     separator=','
# ).filter(
#     (pl.col("sample_library_strategy") == "RNA-Seq") &
#     (pl.col("sample_organism_ch1") == "Homo sapiens") &
#     ~(pl.col("sample_title").str.contains(r'siMETTL14'))
# ).with_columns(
#     (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
#     pl.lit("NEBNext Ultra™ II Directional RNA Library Prep Kit").alias("rna_library"), # add rna_library column from article (New England Biolabs)
#     pl.lit("https://doi.org/10.1038/s41598-024-64517-3").alias("reference_doi") # add DOI of the reference paper column
# ).rename(
#     {
#         "sample_series_id": "gse_id",
#         "big_key": "experiment_accession"
#     }
# ).drop(
#     "sra",
#     "biosample"
# )

# # metadata from iseq
# GSE254232 = pl.read_csv(
#     "../../Data/metadata/metadata_iseq/GSE254232_metadata.tsv",
#     separator='\t'
# ).drop(
#     drop_columns_list
# )

# GSE254232_iseq_filtered = GSE254232[[s.name for s in GSE254232 if not (s.null_count() == GSE254232.height)]]

# GSE254232_combined = GSE254232_PEP.join(
#     GSE254232_iseq_filtered,
#     on=["experiment_accession", "sample_accession"],
#     how="left",
#     coalesce=True
# )

# GSE254232_combined = GSE254232_combined[[s.name for s in GSE254232_combined if not (s.null_count() == GSE254232_combined.height)]]

# GSE254232_combined.write_csv(
#     "../../Data/metadata/metadata_modified/GSE254232_modified_metadata.tsv",
#     separator='\t'
# )

# # display(GSE254232_combined)

In [35]:
# defined_pairs_gse254232 = [
#     ("SRR27726109", "SRR27726111"), # EC109 cell, heat shock 1 vs EC109 cell, si control 1
#     ("SRR27726108", "SRR27726110"), # EC109 cell, heat shock 2 vs EC109 cell, si control 2
#     ("SRR27726107", "SRR27726105"), # EC109 cell, heat shock and siMETTL3 1 vs EC109 cell,  siMETTL3 1
#     ("SRR27726106", "SRR27726104")  # EC109 cell, heat shock and siMETTL3 2 vs EC109 cell,  siMETTL3 2
# ]

# paired_metadata_df_gse254232 = create_paired_metadata(
#     combined_df=GSE254232_combined,
#     defined_sample_pairs=defined_pairs_gse254232,
#     sample_id_column_name="accession" # default value, but specified to clarify
# )

# if paired_metadata_df_gse254232 is not None:
#     paired_metadata_df_gse254232.write_csv(
#         "../../Data/metadata/metadata_paired/GSE254232_paired_metadata.tsv",
#         separator="\t"
#     )

#     display(paired_metadata_df_gse254232)
# else:
#     print("No paired metadata to write (create_paired_metadata returned None).")

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

#### GSE266253 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [DNAJB1-PRKACA Fusion Drives Fibrolamellar Liver Cancer through Impaired SIK Signaling and CRTC2/p300-Mediated Transcriptional Reprogramming](https://doi.org/10.1158/2159-8290.CD-24-0634)
- HSPであるDNAJB1に関連する論文だが、熱ストレスに関連するデータは含まれていないため、今回は収集しない

In [36]:
# GSE266253_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE266253/GSE266253_PEP/GSE266253_PEP_raw.csv",
#     separator=','
# )

# # display(GSE266253_PEP)

#### GSE274408 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Heat shock transcription factor 1 facilitates liver cancer progression by driving super-enhancer-mediated transcription of MYCN](https://doi.org/10.1002/cam4.70157)
- HSF1やMYCNのノックダウンのデータが含まれている
- 熱ストレス関連のデータは含まれていないため、今回は収集しない

In [37]:
# GSE274408_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE274408/GSE274408_PEP/GSE274408_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_library_strategy") == "RNA-Seq"
# )

# # display(GSE274408_PEP)

#### GSE252140 (PAIRED) NO DONWLOAD

- validation date: 2025-05-06
- 論文は見つけられなかった...
- HSPA6に言及されているが、メタデータを見ると熱ストレスに関連したデータではなかったので今回は収集しない

In [38]:
# GSE252140_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE252140/GSE252140_PEP/GSE252140_PEP_raw.csv",
#     separator=','
# )

# # display(GSE252140_PEP)

#### GSE243359 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- 論文は見つけられなかった...
- scRNA-seqのデータで、`heat`と書いてあるが、熱ストレス条件が書いていなく、コントロールもどれだかわからないので今回は収集しない

In [39]:
# GSE243359_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE243359/GSE243359_PEP/GSE243359_PEP_raw.csv",
#     separator=','
# )

# # display(GSE243359_PEP)

#### GSE239482 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Pre-print: Allosteric Inhibition of Proteolytic and ATPase Activities of the Proteasome by the BTK Inhibitor CGI-1746](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4530813)
- treatmentを見ると、`bortezomib` など熱ストレス処理に関連したデータは含まれていないため、今回は収集しない

In [40]:
# GSE239482_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE239482/GSE239482_PEP/GSE239482_PEP_raw.csv",
#     separator=','
# )

# # display(GSE239482_PEP)

#### GSE235379 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Pre-print: The essential chaperone DNAJC17 activates HSP70 to coordinate RNA splicing and G2-M progression](https://doi.org/10.1101/2023.10.25.564066)
- HSPの一つであるDNAJC17に言及されているが、熱ストレスに関連したデータは含まれていないため、今回は収集しない

In [41]:
# GSE235379_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE235379/GSE235379_PEP/GSE235379_PEP_raw.csv",
#     separator=','
# )

# # display(GSE235379_PEP)

#### GSE243360 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- 論文は見つけられなかった
- HSPに関する言及はあるが、メタデータを見ると、はっきり条件がわからないことに加え、熱ストレスに関連したデータが含まれていないため、今回は収集しない

In [42]:
# GSE243360_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE243360/GSE243360_PEP/GSE243360_PEP_raw.csv",
#     separator=','
# )

# # display(GSE243360_PEP)

#### GSE230426 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Transcriptomic identification of genes expressed in invasive S. aureus diabetic foot ulcer infection](https://doi.org/10.3389/fcimb.2023.1198115)
- `Infection in diabetic foot ulcers (DFUs)`に関する研究
- HSPに関する言及はあるものの、メタデータを見ると熱ストレスをかけているデータではないので、今回は収集しない

In [43]:
# GSE230426_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE230426/GSE230426_PEP/GSE230426_PEP_raw.csv",
#     separator=','
# )

# # display(GSE230426_PEP)

### GSE218180 (PAIRED) FANCI

- validation date: 2025-05-06
- [RNA damage compartmentalization by DHX9 stress granules](https://doi.org/10.1016/j.cell.2024.02.028)
- DHX9というヘリカーゼが関与する stress granuleに関する研究
- RNA-seqのデータの他に`fluorescence-activated non-membrane condensates isolation (FANCI)`というSGをフィルタリングする手法も含まれている
- FANCIのサンプルデータに熱ストレス曝露のデータが含まれており, `sample_library_source`を見ると transcriptomicになっており、悩みどころである
- **後でもう一度判断するとして、念の為収集しておく**
- 実験条件は43℃, 60min. (論文より)
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### 5. GSE220877 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-06
- [Maternal nutritional status modifies heat-associated growth restriction in women with chronic malnutrition](https://doi.org/10.1093/pnasnexus/pgac309)
- `Early pregnancy heat exposure impacts the placental gene expression`というようなセクションがあるような熱ストレスと胎児の発育不良に焦点を当てた研究
- `Classification into the HS vs. non-HS groups was determined by the number of HS days (days with Tmax > 39°C) in the first trimester.`という記述から、かなり長期の熱ストレスを受けているサンプルとそうでないサンプルが含まれている
- 扱いが難しい所だが、念の為収集する
- `arm`(今回でいうとグループ分け？), `sex`を揃えた上で、`heatstesst1`がYESかNOかでペアを作成しよう
- とりあえず 1対のペアを作成できたら良しとする (`heatstresst1` = YES のサンプルの方が少ないため、コントロールとするデータは使用しないデータも)
- validation date: 2026-06-26
- 長期の熱ストレスに関する研究であり、今回想定しているような `Heat shock` にあたる実験条件ではないため、結果の解釈を容易にするためにも、今回は収集しないものとする

In [44]:
# GSE220877_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE220877/GSE220877_PEP/GSE220877_PEP_raw.csv",
#     separator=','
# ).with_columns(
#     (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
#     pl.lit("Illumina TruSeq reagents").alias("rna_library"), # add rna_library column from GEO page (Illumina)
#     pl.lit("https://doi.org/10.1093/pnasnexus/pgac309").alias("reference_doi"), # add DOI of the reference paper column
#     pl.lit("Daily maximal temperature > 39 C").alias("stress treatment temperature"), # add heat treatment temperature column
#     pl.lit("More than 20 days within the 90-day window of the first trimester").alias("stress treatment time"), # add heat treatment time column (minutes)
#     pl.lit(None).alias("control treatment temperature"), # add control temperature column
# ).rename(
#     {
#         "sample_series_id": "gse_id",
#         "big_key": "experiment_accession"
#     }
# ).drop(
#     "sra",
#     "biosample"
# ).sort(
#     [
#         "arm",
#         "sex",
#         "heatstesst1",
#         "sample_geo_accession"
#     ],
#     descending=[
#         True,
#         True,
#         True,
#         True
#     ]
# )

# # metadata from iseq
# GSE220877 = pl.read_csv(
#     "../../Data/metadata/metadata_iseq/GSE220877_metadata.tsv",
#     separator='\t'
# ).drop(
#     drop_columns_list
# )

# GSE220877_iseq_filtered = GSE220877[[s.name for s in GSE220877 if not (s.null_count() == GSE220877.height)]]

# GSE220877_combined = GSE220877_PEP.join(
#     GSE220877_iseq_filtered,
#     on=["experiment_accession", "sample_accession"],
#     how="left",
#     coalesce=True
# )

# GSE220877_combined = GSE220877_combined[[s.name for s in GSE220877_combined if not (s.null_count() == GSE220877_combined.height)]]

# GSE220877_combined.write_csv(
#     "../../Data/metadata/metadata_modified/GSE220877_modified_metadata.tsv",
#     separator='\t'
# )

# # display(GSE220877_combined)

In [45]:
# defined_pairs_gse220877 = [
#     # arm:two, sex:male
#     ("SRR22715599", "SRR22715533"), # WF_947380C vs WF_968880L
#     ("SRR22715552", "SRR22715594"), # WF_934580P vs WF_952220U
#     ("SRR22715561", "SRR22715548"), # WF_931450U vs WF_939220U
#     ("SRR22715588", "SRR22715573"), # WF_926020D vs WF_930120A
#     ("SRR22715617", "SRR22715613"), # WF_909180B vs WF_918020W
#     ("SRR22715639", "SRR22715614"), # WF_901480G vs WF_917720W
#     ("SRR22715646", "SRR22715619"), # WF_900720R vs WF_908850K
#     # arm:two, sex:female
#     ("SRR22715611", "SRR22715528"), # WF_918420G vs WF_970220S
#     ("SRR22715627", "SRR22715531"), # WF_905080C vs WF_969220C
#     ("SRR22715651", "SRR22715593"), # WF_900420G vs WF_952820T
#     ("SRR22715652", "SRR22715543"), # WF_900350B vs WF_943580O
#     # arm:three, sex:male
#     ("SRR22715597", "SRR22715598"), # WF_951890D vs WF_951630G
#     ("SRR22715648", "SRR22715603"), # WF_900590E vs WF_946990D
#     # arm:three, sex:female
#     ("SRR22715529", "SRR22715530"), # WF_969530K vs WF_969360J
#     ("SRR22715535", "SRR22715538"), # WF_965760N vs WF_964960K
#     ("SRR22715560", "SRR22715589"), # WF_931560N vs WF_964490V
#     ("SRR22715637", "SRR22715596"), # WF_901960P vs WF_952060Q
#     # arm:one, sex:male
#     ("SRR22715563", "SRR22715527"), # WF_931240I vs WF_970310R
#     ("SRR22715587", "SRR22715537"), # WF_926040U vs WF_965270H
#     # arm:one, sex:female
#     ("SRR22715532", "SRR22715534"), # WF_968940T vs WF_968640I
#     ("SRR22715539", "SRR22715536"), # WF_964740B vs WF_965740T
#     ("SRR22715600", "SRR22715604"), # WF_947340O vs WF_944070J
#     ("SRR22715546", "SRR22715547"), # WF_939410P vs WF_939310T
#     ("SRR22715562", "SRR22715551"), # WF_931270W vs WF_938640D
#     ("SRR22715577", "SRR22715566"), # WF_927140L vs WF_931110V
#     ("SRR22715580", "SRR22715567"), # WF_926810U vs WF_930870D
#     ("SRR22715616", "SRR22715579"), # WF_917170I vs WF_927040P
#     ("SRR22715623", "SRR22715584"), # WF_908540C vs WF_926370W
# ]

# paired_metadata_df_gse220877 = create_paired_metadata(
#     combined_df=GSE220877_combined,
#     defined_sample_pairs=defined_pairs_gse220877,
#     sample_id_column_name="accession" # default value, but specified to clarify
# )

# if paired_metadata_df_gse220877 is not None:
#     paired_metadata_df_gse220877.write_csv(
#         "../../Data/metadata/metadata_paired/GSE220877_paired_metadata.tsv",
#         separator='\t'
#     )

#     display(paired_metadata_df_gse220877)
# else:
#     print("No paired metadata to write (create_paired_metadata returned None).")

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

#### GSE209863 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Heat Shock Protein Family A Member 1 Promotes Intracellular Amplification of Hepatitis B Virus Covalently Closed Circular DNA](https://doi.org/10.1128/jvi.01261-22)
- `B02(a small-molecule inhibitor of DNA homologous recombination repair protein RAD51)`という阻害剤を用いた解析
- HSPに言及はされているが、今回は収集しない

In [46]:
# GSE209863_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE209863/GSE209863_PEP/GSE209863_PEP_raw.csv",
#     separator=','
# )

# # display(GSE209863_PEP)

#### GSE182851 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [FOXR1 regulates stress response pathways and is necessary for proper brain development](https://doi.org/10.1371/journal.pgen.1009854)
- heat shock response (HSR)に関連があると言われているFOXR1に関する研究
- 熱ショックを加えているデータは含まれていないので今回は収集しない

In [47]:
# GSE182851_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE182851/GSE182851_PEP/GSE182851_PEP_raw.csv",
#     separator=','
# )

# # display(GSE182851_PEP)

#### GSE164742 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Heat shock protein 60 couples an oxidative stress-responsive p38/MK2 signaling and NF-κB survival machinery in cancer cells](https://doi.org/10.1016/j.redox.2022.102293)
- HSP60と酸化ストレスの関係について研究している論文
- treatmentを見ると過酸化水素曝露処理のサンプルが含まれており、熱ストレス関連のデータがないため、今回は収集しない

In [48]:
# GSE164742_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE164742/GSE164742_PEP/GSE164742_PEP_raw.csv",
#     separator=','
# )

# # display(GSE164742_PEP)

#### GSE205596 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Proteotoxic stress-induced apoptosis in cancer cells: understanding the susceptibility and enhancing the potency](https://doi.org/10.1038/s41420-022-01202-2)
- Proteotoxic stress に関連する `2c`という化合物に関連する処理を `treatment`カラムを見ると行われている
- ただ、熱ストレスに関連したデータはなさそうなので、今回は収集しない

In [49]:
# GSE205596_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE205596/GSE205596_PEP/GSE205596_PEP_raw.csv",
#     separator=','
# )

# # display(GSE205596_PEP)

#### GSE212119 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Transcriptomic profiling of cardiac tissues from SARS-CoV-2 patients identifies DNA damage](https://doi.org/10.1111/imm.13577)
- 論文、メタデータを見ても熱ストレスに関連したデータはなさそうなので今回は収集しない

In [50]:
# GSE212119_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE212119/GSE212119_PEP/GSE212119_PEP_raw.csv",
#     separator=','
# )

# # display(GSE212119_PEP)

#### GSE163102 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Carfilzomib Treatment Causes Molecular and Functional Alterations of Human Induced Pluripotent Stem Cell–Derived Cardiomyocytes](https://doi.org/10.1161/JAHA.121.022247)
- Carfilzomibを処理しているデータで、熱ストレス処理をしているデータは含まれていなかったので今回は収集しない

In [51]:
# GSE163102_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE163102/GSE163102_PEP/GSE163102_PEP_raw.csv",
#     separator=','
# )

# # display(GSE163102_PEP)

#### GSE171440 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Cytoplasmic proteotoxicity regulates HRI-dependent phosphorylation of eIF2α via the Hsp70-Bag3 module](https://doi.org/10.1016/j.isci.2022.104282)
- `Treated with JG-98 for 12hours`のサンプルとCtrlのサンプルが登録されている
- 熱ストレスに関連したデータは登録されていないので、今回は収集しない

In [52]:
# GSE171440_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE171440/GSE171440_PEP/GSE171440_PEP_raw.csv",
#     separator=','
# )

# # display(GSE171440_PEP)

#### GSE151514 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [17-DMAG dually inhibits Hsp90 and histone lysine demethylases in alveolar rhabdomyosarcoma](https://doi.org/10.1016/j.isci.2020.101996)
- Histone lysine demethylases (KDMs) の阻害剤として発見された17-DMAGについて調べている論文
- HSPに言及はされているが、熱ストレスに関連したデータは含まれていないため、今回は収集しない

In [53]:
# GSE151514_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE151514/GSE151514_PEP/GSE151514_PEP_raw.csv",
#     separator=','
# )

# # display(GSE151514_PEP)

#### GSE134315 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Rho-Associated Coiled-Coil Kinase 1 Translocates to the Nucleus and Inhibits Human Cytomegalovirus Propagation](https://doi.org/10.1128/jvi.00453-19)
- ウイルスに感染したサンプルに関するデータが登録されており、熱ストレスに関連しているデータは登録されていないので、今回は収集しない

In [54]:
# GSE134315_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE134315/GSE134315_PEP/GSE134315_PEP_raw.csv",
#     separator=','
# )

# # display(GSE134315_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### 6. GSE127344 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-06
- [In Vivo Developmental Trajectories of Human Podocyte Inform In Vitro Differentiation of Pluripotent Stem Cell-Derived Podocytes](https://doi.org/10.1016/j.devcel.2019.06.001)
- bulk RNA-seqのデータ
- thermoswitchという表現がされている
- `heat shock activation: Cultured at 37C`と `heat shock activation: Cultured at 33C`のデータを収集しよう
- validate date: 2026-06-26
- 曝露時間が記載されていないため、今回は収集しないものとする


In [55]:
# GSE127344_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE127344/GSE127344_PEP/GSE127344_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_library_strategy") == "RNA-Seq"
# ).filter(
#     pl.col("sample_title").str.contains(r"immortal.")
# ).with_columns(
#     (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
#     pl.lit("Kapa stranded mRNA-Seq library").alias("rna_library"), # add rna_library column from GEO page (New England Biolabs)
#     pl.lit("https://doi.org/10.1016/j.devcel.2019.06.001").alias("reference_doi"), # add DOI of the reference paper column
#     pl.lit("37 C?").alias("stress treatment temperature"), # add heat treatment temperature column
#     pl.lit(None).alias("stress treatment time"), # add heat treatment time column (minutes)
#     pl.lit("33 C?").alias("control treatment temperature"), # add control temperature column
# ).rename(
#     {
#         "sample_series_id": "gse_id",
#         "big_key": "experiment_accession"
#     }
# ).drop(
#     "sra",
#     "biosample"
# )

# # metadata from iseq
# GSE127344 = pl.read_csv(
#     "../../Data/metadata/metadata_iseq/GSE127344_metadata.tsv",
#     separator='\t'
# ).drop(
#     drop_columns_list
# )

# GSE127344_iseq_filtered = GSE127344[[s.name for s in GSE127344 if not (s.null_count() == GSE127344.height)]]

# GSE127344_combined = GSE127344_PEP.join(
#     GSE127344_iseq_filtered,
#     on=["experiment_accession", "sample_accession"],
#     how="left",
#     coalesce=True
# )

# GSE127344_combined = GSE127344_combined[[s.name for s in GSE127344_combined if not (s.null_count() == GSE127344_combined.height)]]

# GSE127344_combined.write_csv(
#     "../../Data/metadata/metadata_modified/GSE127344_modified_metadata.tsv",
#     separator='\t'
# )

# # display(GSE127344_combined)

In [56]:
# defined_pairs_gse127344 = [
#     ("SRR9171830", "SRR9171828"), # immortal.pod.post-thermoswitch.r1 vs immortal.pod.pre-thermoswitch.r1
#     ("SRR9171831", "SRR9171829"), # immortal.pod.post-thermoswitch.r2 vs immortal.pod.pre-thermoswitch.r2
#     ("SRR9171832", "SRR9171829"), # immortal.pod.post-thermoswitch.r2 vs immortal.pod.pre-thermoswitch.r2
#     ("SRR9171833", "SRR9171829")  # immortal.pod.post-thermoswitch.r2 vs immortal.pod.pre-thermoswitch.r2
# ]

# paired_metadata_df_gse127344 = create_paired_metadata(
#     combined_df=GSE127344_combined,
#     defined_sample_pairs=defined_pairs_gse127344,
#     sample_id_column_name="accession" # default value, but specified to clarify
# )

# if paired_metadata_df_gse127344 is not None:
#     paired_metadata_df_gse127344.write_csv(
#         "../../Data/metadata/metadata_paired/GSE127344_paired_metadata.tsv",
#         separator='\t'
#     )

#     display(paired_metadata_df_gse127344)
# else:
#     print("No paired metadata to write (create_paired_metadata returned None).")

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

#### GSE108622 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Mitochondrial carrier protein overloading and misfolding induce aggresomes and proteostatic adaptations in the cytosol](https://doi.org/10.1091/mbc.E19-01-0046)
- ANT1という遺伝子に関連しているデータで、熱ストレスに関連したデータは含まれていないので今回は収集しない


In [57]:
# GSE108622_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE108622/GSE108622_PEP/GSE108622_PEP_raw.csv",
#     separator=','
# )

# # display(GSE108622_PEP)

#### GSE116927 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [HSP90 inhibitor geldanamycin reverts IL-13– and IL-17–induced airway goblet cell metaplasia](https://doi.org/10.1172/JCI123524)
- HSP90の阻害剤である geldanamycinに関するデータ
- メタデータを見てみたところ、熱ストレスに関連したデータはなさそうだったので、今回は収集しない


In [58]:
# GSE116927_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE116927/GSE116927_PEP/GSE116927_PEP_raw.csv",
#     separator=','
# )

# # display(GSE116927_PEP)

#### GSE67041 (PAIRED)

- validation date: 2025-05-06
- [RNA polymerase II pausing can be retained or acquired during activation of genes involved in the epithelial to mesenchymal transition](https://doi.org/10.1093/nar/gkv263)
- RNA-seqとChIP-seqのデータが含まれてあるデータ
- short capped RNA に関連するデータ
- 熱ストレスに関連するデータは含まれていないので今回は収集しない


In [59]:
# GSE67041_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE67041/GSE67041_PEP/GSE67041_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("protocol") == "cDNA"
# )

# # display(GSE67041_PEP)

#### GSE60047 (PAIRED) NO DOWNLOAD

- validation date: 2025-05-06
- [Transcriptome-wide Mapping Reveals Widespread Dynamic-Regulated Pseudouridylation of ncRNA and mRNA](https://doi.org/10.1016/j.cell.2014.08.028)
- シュードウリジンを用いたsequence 手法に関連する方法
- ヒトのデータのみをフィルタリングしたとき、コントロールと `N3-[N-cyclohexyl-N’-β-(4-methylmorpholinium)ethylcarbodiimide-Ψ (N3-CMC-Ψ) `処理を加えている論文
- 熱ストレスを加えているサンプルは今回ヒトに関連しているデータにはないので、今回は収集しない


In [60]:
# GSE60047_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE60047/GSE60047_PEP/GSE60047_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_organism_ch1") == "Homo sapiens"
# )

# # display(GSE60047_PEP)

#### GSE50445 (PAIRED)

- validation date: 2025-05-06
- [IVT-seq reveals extreme bias in RNA sequencing](https://doi.org/10.1186/gb-2014-15-6-r86)
- `in vitro transcription sequencing (IVT-seq)`という手法
- ヒトのデータをフィルタリングすると、熱ストレスに関連したデータはどうやら含まれていないっぽいので今回は収集しない

In [61]:
# GSE50445_PEP = pl.read_csv(
#     "../../Data/metadata/metadata_geofetch/GSE50445/GSE50445_PEP/GSE50445_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_organism_ch1") == "Homo sapiens"
# )

# # display(GSE50445_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Concat dataframe (RNA-seq only)

In [62]:
dfs_rna_seq_false_no_control_all: list[pl.DataFrame | None] = [
    paired_metadata_df_gse124510,
    paired_metadata_df_gse194098,
    paired_metadata_df_gse180253_rna_seq,
    paired_metadata_df_gse145525,
    # paired_metadata_df_gse220877,
    # paired_metadata_df_gse127344
]

dfs_rna_seq_false_no_control: list[pl.DataFrame] = [df for df in dfs_rna_seq_false_no_control_all if df is not None]

In [63]:
rna_seq_false_no_control = pl.concat(
    dfs_rna_seq_false_no_control,
    how="diagonal_relaxed"
)

rna_seq_false_no_control.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_false_no_control_metadata_{execute_date}.tsv",
    separator="\t"
)